# SemCor Cross-Encoder Evaluation

这个 notebook 复用 `semcor_embeds_explore.ipynb` 的 SemCor 读取方式，并把每个带 `synset_name`/`synset_definition` 的名词标注转换成 cross-encoder 重排任务。

评测思路：先固定一个目标词，收集它在 SemCor 中出现的全部句子样本和全部候选 synset definition。对于每一句样本，先构造 `Sentence / Target word / Question` 形式的 prompt，再把每个 definition 改写成 `It refers to ...` 的 hypothesis，逐一配对打分，统计 Top-1 Accuracy 与 MRR。

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from itertools import islice
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder

from prepare_semcor import iter_sentence_records, load_semcor_stats


In [3]:
BASE_DIR = Path("data/semcor")
NUM_SENTENCES = 37176
DEFAULT_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
DEFAULT_RANDOM_SEED = 13
TARGET_WORD = "house"

def load_semcor_sentence_sample(base_dir: Path, num_sentences: int | None = None):
    records = iter_sentence_records(base_dir)
    if num_sentences is None:
        return list(records)
    return list(islice(records, num_sentences))


stats = load_semcor_stats(BASE_DIR)
sentence_records = load_semcor_sentence_sample(BASE_DIR, NUM_SENTENCES)

print(f"Loaded {len(sentence_records)} SemCor sentences")
pd.Series(stats)


Loaded 37176 SemCor sentences


dataset_name                                                             SemCor
base_dir                                   /home/xiaoyue/LiteSemRAG/data/semcor
nltk_data_dir                 /home/xiaoyue/LiteSemRAG/data/semcor/raw/nltk_...
processed_dir                    /home/xiaoyue/LiteSemRAG/data/semcor/processed
resources                                            [semcor, wordnet, omw-1.4]
file_count                                                                  352
sentence_count                                                            37176
token_count                                                              820410
annotation_count                                                         778587
semantic_annotation_count                                                235079
noun_annotation_count                                                     88892
oov_entity_count                                                           9684
multiword_annotation_count              

In [4]:
def build_token_char_spans(sentence_text: str, tokens: list[str]) -> list[tuple[int, int]]:
    spans = []
    cursor = 0

    for token in tokens:
        while cursor < len(sentence_text) and sentence_text[cursor].isspace():
            cursor += 1

        start_char = sentence_text.find(token, cursor)
        if start_char < 0:
            raise ValueError(
                f"Could not align token {token!r} inside sentence starting from char {cursor}.\n"
                f"text={sentence_text!r}"
            )

        gap_text = sentence_text[cursor:start_char]
        if any(not char.isspace() for char in gap_text):
            raise ValueError(
                f"Unexpected non-space gap {gap_text!r} before token {token!r}.\n"
                f"text={sentence_text!r}"
            )

        end_char = start_char + len(token)
        spans.append((start_char, end_char))
        cursor = end_char

    return spans


def mark_target_in_sentence(
    sentence_record: dict,
    annotation: dict,
    left_marker: str = "[TGT]",
    right_marker: str = "[/TGT]",
) -> str:
    token_spans = build_token_char_spans(sentence_record["text"], sentence_record["tokens"])
    start_char = token_spans[annotation["token_start"]][0]
    end_char = token_spans[annotation["token_end"] - 1][1]
    sentence_text = sentence_record["text"]
    target_text = sentence_text[start_char:end_char]
    return f"{sentence_text[:start_char]}{left_marker} {target_text} {right_marker}{sentence_text[end_char:]}"


def extract_semcor_cross_encoder_examples(
    sentence_records: list[dict],
    target_word: str | None = None,
    max_examples: int | None = None,
    mark_target: bool = False,
) -> list[dict]:
    examples = []
    normalized_target_word = None if target_word is None else target_word.strip().lower()

    for sentence_record in sentence_records:
        for annotation in sentence_record.get("noun_annotations", []):
            synset_name = annotation.get("synset_name")
            synset_definition = annotation.get("synset_definition")
            if not synset_name or not synset_definition:
                continue

            surface_text = " ".join(annotation.get("tokens", []))
            lemma = annotation.get("lemma")
            if (lemma or "").strip().lower() == "group" or synset_name == "group.n.01":
                continue
            if normalized_target_word is not None:
                candidate_terms = {
                    surface_text.strip().lower(),
                    (lemma or "").strip().lower(),
                }
                if normalized_target_word not in candidate_terms:
                    continue

            target_token = target_word if target_word is not None else surface_text
            query_text = sentence_record["text"]
            if mark_target:
                query_text = mark_target_in_sentence(sentence_record, annotation)
            prompt_text = (
                f"Sentence: {query_text}\n"
                f"Target word: {target_token}\n\n"
                f'Question: What does "{target_token}" mean in this sentence?'
            )

            examples.append(
                {
                    "sentence_id": sentence_record["sentence_id"],
                    "sentence_text": sentence_record["text"],
                    "query_text": prompt_text,
                    "prompt_text": prompt_text,
                    "surface_text": surface_text,
                    "lemma": lemma,
                    "synset_name": synset_name,
                    "synset_definition": synset_definition,
                }
            )

            if max_examples is not None and len(examples) >= max_examples:
                return examples

    return examples


def definition_to_hypothesis(definition: str) -> str:
    cleaned_definition = definition.strip()
    if cleaned_definition.endswith((".", "!", "?")):
        cleaned_definition = cleaned_definition[:-1]
    return f"It refers to {cleaned_definition}."


def extract_cross_encoder_scores(raw_scores, model) -> np.ndarray:
    score_array = np.asarray(raw_scores)
    if score_array.ndim == 1:
        return score_array.astype(float)

    id2label = getattr(model.model.config, "id2label", {}) or {}
    entailment_index = None
    for label_index, label_name in id2label.items():
        if str(label_name).lower() == "entailment":
            entailment_index = int(label_index)
            break

    if entailment_index is None:
        entailment_index = score_array.shape[1] - 1

    return score_array[:, entailment_index].astype(float)


def build_synset_candidate_bank(examples: list[dict]) -> list[dict]:
    candidate_map = {}
    for example in examples:
        candidate_map.setdefault(
            example["synset_name"],
            {
                "synset_name": example["synset_name"],
                "definition": example["synset_definition"],
                "hypothesis": definition_to_hypothesis(example["synset_definition"]),
            },
        )
    return list(candidate_map.values())


In [5]:
example_rows = extract_semcor_cross_encoder_examples(
    sentence_records,
    target_word=TARGET_WORD,
    max_examples=5,
    mark_target=False,
)
candidate_bank = build_synset_candidate_bank(
    extract_semcor_cross_encoder_examples(
        sentence_records,
        target_word=TARGET_WORD,
        max_examples=None,
        mark_target=False,
    )
)

print(f"Target word: {TARGET_WORD}")
print(f"Candidate bank size: {len(candidate_bank)} unique synsets")
pd.DataFrame(example_rows)[
    [
        "sentence_id",
        "surface_text",
        "lemma",
        "synset_name",
        "synset_definition",
        "query_text",
    ]
]


Target word: house
Candidate bank size: 8 unique synsets


,sentence_id,surface_text,lemma,synset_name,synset_definition,query_text
0,brown1/tagfiles/br-a02.xml:44,House,house,house.n.05,an official assembly having legislative powers,"Sentence: Under Formby's plan, an appointee wo..."
1,brown1/tagfiles/br-a11.xml:19,House,person,person.n.01,a human being,"Sentence: Pete Ward was sent in for House and,..."
2,brown1/tagfiles/br-c01.xml:27,house,house,house.n.04,the audience gathered together in a theatre or...,Sentence: The engagement was supposed to be al...
3,brown1/tagfiles/br-c01.xml:93,house,house,house.n.01,a dwelling that serves as living quarters for ...,Sentence: He doesn't think that potting them f...
4,brown1/tagfiles/br-d03.xml:60,houses,house,house.n.03,the members of a religious community living to...,"Sentence: Now, not only are there considerably..."


In [6]:
def evaluate_cross_encoder_on_semcor(
    target_word: str,
    model_name: str = DEFAULT_MODEL_NAME,
    sentence_records: list[dict] | None = None,
    candidate_sentence_records: list[dict] | None = None,
    mark_target: bool = False,
    batch_size: int = 32,
):
    if sentence_records is None:
        sentence_records = load_semcor_sentence_sample(BASE_DIR, NUM_SENTENCES)

    if candidate_sentence_records is None:
        candidate_sentence_records = sentence_records

    examples = extract_semcor_cross_encoder_examples(
        sentence_records=sentence_records,
        target_word=target_word,
        max_examples=None,
        mark_target=mark_target,
    )
    if not examples:
        raise ValueError(f"No SemCor examples were found for target_word={target_word!r}.")

    candidate_bank = build_synset_candidate_bank(
        extract_semcor_cross_encoder_examples(
            sentence_records=candidate_sentence_records,
            target_word=target_word,
            max_examples=None,
            mark_target=False,
        )
    )
    if len(candidate_bank) < 2:
        raise ValueError(
            f"Need at least two distinct synsets for target_word={target_word!r} to evaluate disambiguation."
        )

    model = CrossEncoder(model_name)

    top1_correct = 0
    reciprocal_ranks = []
    evaluation_rows = []

    for example in examples:
        candidates = [
            {
                "synset_name": candidate["synset_name"],
                "definition": candidate["definition"],
                "hypothesis": candidate["hypothesis"],
                "label": int(candidate["synset_name"] == example["synset_name"]),
            }
            for candidate in candidate_bank
        ]

        pairs = [(example["query_text"], candidate["hypothesis"]) for candidate in candidates]
        raw_scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        scores = extract_cross_encoder_scores(raw_scores, model)

        ranked_candidates = sorted(
            [
                {
                    **candidate,
                    "score": float(score),
                }
                for candidate, score in zip(candidates, scores)
            ],
            key=lambda item: item["score"],
            reverse=True,
        )

        gold_rank = next(
            rank for rank, candidate in enumerate(ranked_candidates, start=1) if candidate["label"] == 1
        )
        top_candidate = ranked_candidates[0]

        top1_correct += int(gold_rank == 1)
        reciprocal_ranks.append(1.0 / gold_rank)
        evaluation_rows.append(
            {
                "sentence_id": example["sentence_id"],
                "surface_text": example["surface_text"],
                "lemma": example["lemma"],
                "gold_synset_name": example["synset_name"],
                "gold_definition": example["synset_definition"],
                "gold_rank": gold_rank,
                "is_top1_correct": gold_rank == 1,
                "top_prediction": top_candidate["synset_name"],
                "top_prediction_definition": top_candidate["definition"],
                "top_prediction_hypothesis": top_candidate["hypothesis"],
                "top_prediction_score": top_candidate["score"],
                "query_text": example["query_text"],
            }
        )

    results_df = pd.DataFrame(evaluation_rows)
    metrics = pd.Series(
        {
            "target_word": target_word,
            "model_name": model_name,
            "num_examples": len(examples),
            "num_unique_synsets": len(candidate_bank),
            "candidate_synsets_per_example": len(candidate_bank),
            "top1_accuracy": float(top1_correct / len(examples)),
            "mrr": float(np.mean(reciprocal_ranks)),
            "mean_gold_rank": float(results_df["gold_rank"].mean()),
        }
    )
    return metrics, results_df


In [9]:
TARGET_WORD = "face"
metrics, results_df = evaluate_cross_encoder_on_semcor(
    target_word=TARGET_WORD,
    model_name=DEFAULT_MODEL_NAME,
    sentence_records=sentence_records,
    mark_target=False,
)

metrics


target_word                                                   face
model_name                       cross-encoder/nli-deberta-v3-base
num_examples                                                   144
num_unique_synsets                                               6
candidate_synsets_per_example                                    6
top1_accuracy                                               0.0625
mrr                                                       0.325579
mean_gold_rank                                            3.701389
dtype: object

In [10]:
results_df.sort_values(["is_top1_correct", "gold_rank"], ascending=[True, False]).head(100)


,sentence_id,surface_text,lemma,gold_synset_name,gold_definition,gold_rank,is_top1_correct,top_prediction,top_prediction_definition,top_prediction_hypothesis,top_prediction_score,query_text
6,brown1/tagfiles/br-j03.xml:49,face,face,face.n.04,the striking or working surface of an implement,6,False,face.n.05,a part of a person that is used to refer to a ...,It refers to a part of a person that is used t...,-2.345748,Sentence: The concept of the strain energy as ...
7,brown1/tagfiles/br-j03.xml:52,face,face,face.n.04,the striking or working surface of an implement,6,False,face.n.05,a part of a person that is used to refer to a ...,It refers to a part of a person that is used t...,-2.366071,Sentence: From this and the force of deformati...
46,brown1/tagfiles/br-k15.xml:53,face,face,face.n.01,the front of the human head from the forehead ...,6,False,face.n.03,the general outward appearance of something,It refers to the general outward appearance of...,-1.166622,Sentence: She had taken him out of the schoolh...
3,brown1/tagfiles/br-g15.xml:55,face,face,face.n.01,the front of the human head from the forehead ...,5,False,face.n.03,the general outward appearance of something,It refers to the general outward appearance of...,-0.823749,Sentence: Piepsam calls the cyclist ``cur'' an...
8,brown1/tagfiles/br-j55.xml:23,face,face,face.n.01,the front of the human head from the forehead ...,5,False,face.n.03,the general outward appearance of something,It refers to the general outward appearance of...,-1.205947,"Sentence: Bullets were so thick, throwing sand..."
...,...,...,...,...,...,...,...,...,...,...,...,...
143,brown2/tagfiles/br-p24.xml:119,face,face,face.n.01,the front of the human head from the forehead ...,4,False,face.n.05,a part of a person that is used to refer to a ...,It refers to a part of a person that is used t...,-1.540880,Sentence: Mike had a good two inches over Phil...
1,brown1/tagfiles/br-c04.xml:91,faces,face,face.n.01,the front of the human head from the forehead ...,3,False,face.n.03,the general outward appearance of something,It refers to the general outward appearance of...,-0.821190,"Sentence: Most of the female faces are new, or..."
4,brown1/tagfiles/br-j03.xml:4,faces,face,face.n.04,the striking or working surface of an implement,3,False,face.n.03,the general outward appearance of something,It refers to the general outward appearance of...,-1.620985,Sentence: A tape of cellulose acetate is pulle...
5,brown1/tagfiles/br-j03.xml:12,face,face,face.n.04,the striking or working surface of an implement,3,False,face.n.03,the general outward appearance of something,It refers to the general outward appearance of...,-1.245527,Sentence: The face of one block contained a ho...
